In [57]:
import torch
import lightning.pytorch as pl

In [58]:
import time

class Timer():
    def __init__(self):
        pass
    
    def __enter__(self):
        self.start_time = time.time()
        
    def __exit__(self, exc_type, exc_val, exc_tb):
        print(time.time() - self.start_time)

In [59]:
# x = torch.randn((3, 3))
x = torch.randint(-15, 15, (3,3))
print(x)

tensor([[  1,  -2,   3],
        [-11,   0,  11],
        [-14,  -3,   1]])


## BReLU 기본형

In [60]:
class BReLU(pl.LightningModule):
    def __init__(self):
        super(BReLU, self).__init__()
        
    def forward(self, x):
        epsilon = torch.distributions.bernoulli.Bernoulli(logits=x).sample()
        return x * epsilon.to(self.device)

In [61]:
brelu = BReLU()
brelu(x)

tensor([[ 1., -0.,  3.],
        [-0.,  0., 11.],
        [-0., -0.,  1.]])

In [62]:
print(brelu)

BReLU()


## Leaky BReLU

In [63]:
class Leaky_BReLU(pl.LightningModule):
    def __init__(self):
        super(Leaky_BReLU, self).__init__()
        
    def forward(self, x):
        epsilon = torch.distributions.bernoulli.Bernoulli(logits=x).sample()
        epsilon = torch.where(epsilon==0, 0.1, 1.0)
        
        return x * epsilon.to(self.device)

In [64]:
leaky = Leaky_BReLU()
leaky(x)

tensor([[ 0.1000, -0.2000,  3.0000],
        [-1.1000,  0.0000, 11.0000],
        [-1.4000, -0.3000,  1.0000]])

In [65]:
print(leaky)

Leaky_BReLU()


## 점근 속도를 조절하는 BReLU

In [66]:
class VariableBReLU(pl.LightningModule):
    def __init__(self, alpha=1):
        super(VariableBReLU, self).__init__()
        self.alpha = alpha
        
    def forward(self, x):
        epsilon = torch.distributions.bernoulli.Bernoulli(logits=self.alpha * x).sample()
        
        return x * epsilon.to(self.device)
    
    def extra_repr(self) -> str:
        return 'alpha={}'.format(self.alpha)

In [67]:
def mv(**kwargs):
    if 'alpha' in list(kwargs.keys()):
        return VariableBReLU(alpha=kwargs.get('alpha'))

mv(alpha=.1)

VariableBReLU(alpha=0.1)

In [68]:
alpha = 0.1
vb = VariableBReLU(alpha=alpha)
vb(x)

tensor([[ 0., -2.,  3.],
        [-0.,  0., 11.],
        [-0., -0.,  1.]])

In [69]:
print(vb)

VariableBReLU(alpha=0.1)


## 결과값을 0아니면 1로 만들기

In [70]:
class MeasureAct(pl.LightningModule):
    def __init__(self):
        super(MeasureAct, self).__init__()
        
    def forward(self, x):
        epsilon = torch.distributions.bernoulli.Bernoulli(logits=x).sample()
        epsilon = epsilon * (1/(x + 1e-16))
        
        return x * epsilon.to(self.device)

In [71]:
ma = MeasureAct()
ma(x)

tensor([[0., 0., 1.],
        [0., 0., 1.],
        [0., 0., 1.]])

## 일부 원소만 0 아니면 1로 만들기

In [72]:
class SomeMeasureAct(pl.LightningModule):
    def __init__(self):
        super(SomeMeasureAct, self).__init__()
        
    def forward(self, x):
        b = torch.distributions.bernoulli.Bernoulli(logits=x)
        b1 = b.sample()
        b2 = b.sample()

        epsilon = b1*b2*(1/(x+1e-16)) + torch.logical_xor(b1, b2)
        
        return x * epsilon

In [73]:
sma = SomeMeasureAct()
sma(x)

tensor([[1., -0., 1.],
        [-0., 0., 1.],
        [-0., -0., 1.]])

## 확률적으로 ReLU, BReLU를 적용하기

In [74]:
class SARB(pl.LightningModule):
    def __init__(self, zero=BReLU(), one=torch.nn.ReLU(inplace=True)):
        super(SARB, self).__init__() #Stochastic Activation Relu or Brelu
        self.zero=zero
        self.one=one
        
    def forward(self, x):
        mu = x.to(torch.float32).mean()
        b = torch.distributions.bernoulli.Bernoulli(logits=mu).sample()
        # print('mu=%d'%mu)
        # print('b=%d'%b)
        
        if b == 0:
            return self.zero(x)
        
        else:
            return self.one(x)
        
    # def extra_repr(self) -> str:
    #     return 'zero={}, one={}'.format(self.zero, self.one)
        

In [75]:
class SARBSC(pl.LightningModule):                         #Stochastic Activation Relu or Brelu(확률적으로 Relu나 BRelu를 적용)
    def __init__(self, zero=BReLU(), one=torch.nn.ReLU(inplace=True)):
        super(SARBSC, self).__init__() 
        self.zero=zero
        self.one=one
        
    def forward(self, x):
        mu = x.to(torch.float32).mean()
        p = -torch.sin(mu) * torch.cos(mu + (torch.pi/2))
        p = torch.where(p < 0, 0, p)
        p = torch.where(p > 1, 1, p)
        # b = torch.distributions.bernoulli.Bernoulli(logits=mu).sample()
        b = torch.distributions.bernoulli.Bernoulli(probs=p).sample()
        # print('mu=%d'%mu)
        # print('b=%d'%b)
        
        if b == 0:
            return self.zero(x)
        
        else:
            return self.one(x)

In [76]:
class BatchSC(pl.LightningModule):                         #Stochastic Activation Relu or Brelu(확률적으로 Relu나 BRelu를 적용)
    def __init__(self, zero=BReLU(), one=torch.nn.ReLU(inplace=True)):
        super(BatchSC, self).__init__() 
        self.zero=zero
        self.one=one
        
    def forward(self, x):
        tmp = []
        mu = x.to(torch.float32).mean(dim=(1,2,3), keepdim=False)
        p = -torch.sin(mu) * torch.cos(mu + (torch.pi/2))
        p = torch.where(p < 0, 0, p)
        p = torch.where(p > 1, 1, p)
        # b = torch.distributions.bernoulli.Bernoulli(logits=mu).sample()
        switch = torch.distributions.bernoulli.Bernoulli(probs=p).sample()
        # print('mu=%d'%mu)
        # print('b=%d'%b)
        
        for b, batch in zip(switch, x):
            if b == 0:
                tmp.append(self.zero(batch))
            
            else:
                tmp.append(self.one(batch))
                
        return torch.stack(tmp)

In [77]:
sarb = SARB()
sarb(x)

tensor([[ 1., -2.,  3.],
        [-0.,  0., 11.],
        [-0., -0.,  1.]])

In [78]:
sarb

SARB(
  (zero): BReLU()
  (one): ReLU(inplace=True)
)

배치별로 다른 활성화함수 적용 또는 채널별

In [79]:
class BatchWiseSARB(pl.LightningModule):
    def __init__(self, zero=BReLU(), one=torch.nn.ReLU(inplace=False)):
        super(BatchWiseSARB, self).__init__() #Stochastic Activation Relu or Brelu
        self.zero=zero
        self.one=one
        
    def forward(self, x):
        tmp = []
        mu = x.to(torch.float32).mean(dim=(1,2,3), keepdim=False)
        switch = torch.distributions.bernoulli.Bernoulli(logits=mu).sample()
        # print('mu=%d'%mu)
        # print('b=%d'%b)
        
        for b, batch in zip(switch, [x[j] for j in range(x.size(0))]):
            if b == 0:
                # tmp.append(self.zero(batch.clone()))
                tmp.append(self.zero(batch))
            
            else:
                # tmp.append(self.one(batch.clone()))
                tmp.append(self.one(batch))
        return torch.stack(tmp)
    

class BatchWiseSARBSC(pl.LightningModule):
    def __init__(self, zero=BReLU(), one=torch.nn.ReLU(inplace=False)):
        super(BatchWiseSARBSC, self).__init__() #Stochastic Activation Relu or Brelu
        self.zero=zero
        self.one=one
        
    def forward(self, x):
        tmp = []
        mu = x.to(torch.float32).mean(dim=(1,2,3), keepdim=False)
        p = -torch.sin(mu) * torch.cos(mu + (torch.pi/2))
        p = torch.where(p < 0, 0, p)
        p = torch.where(p > 1, 1, p)
        switch = torch.distributions.bernoulli.Bernoulli(logits=mu).sample()
        # print('mu=%d'%mu)
        # print('b=%d'%b)
        
        for b, batch in zip(switch, [x[j] for j in range(x.size(0))]):
            if b == 0:
                # tmp.append(self.zero(batch.clone()))
                tmp.append(self.zero(batch))
            
            else:
                # tmp.append(self.one(batch.clone()))
                tmp.append(self.one(batch))
        return torch.stack(tmp)

In [80]:
class ImproveBatchWiseSARBSC(pl.LightningModule):
    def __init__(self, zero=BReLU(), one=torch.nn.ReLU(inplace=True)):
        super(ImproveBatchWiseSARBSC, self).__init__() #Stochastic Activation Relu or Brelu
        self.zero=zero
        self.one=one
        
    def forward(self, x):
        mu = x.to(torch.float32).mean(dim=(1,2,3), keepdim=True)
        p = -torch.sin(mu) * torch.cos(mu + (torch.pi/2))
        p = torch.where(p < 0, 0, p)
        p = torch.where(p > 1, 1, p)
        b = torch.distributions.bernoulli.Bernoulli(probs=p).sample()
        # print('mu=%d'%mu)
        # print('b=%d'%b)
        
        return self.zero((1-b) * x) + self.one(b * x)

In [81]:
testX = torch.rand((256, 2, 2, 2))

In [82]:
mu = testX.mean(dim=(1,2,3), keepdim=True)
b = torch.distributions.bernoulli.Bernoulli(logits=mu).sample()

In [83]:
b.size()

torch.Size([256, 1, 1, 1])

In [84]:
testX * (1-b) + testX * b

tensor([[[[7.5499e-01, 1.7076e-01],
          [6.7209e-01, 7.8012e-02]],

         [[4.8203e-01, 8.6753e-01],
          [3.3811e-01, 3.0815e-01]]],


        [[[5.9437e-01, 6.8330e-01],
          [5.3126e-01, 8.6268e-01]],

         [[9.1916e-01, 1.6743e-01],
          [7.4843e-01, 2.5929e-01]]],


        [[[2.4428e-01, 9.6529e-01],
          [1.7327e-01, 2.1513e-01]],

         [[8.3670e-01, 8.0519e-01],
          [5.1343e-01, 6.7952e-01]]],


        ...,


        [[[5.4363e-01, 6.2376e-01],
          [6.4436e-01, 9.3477e-01]],

         [[2.2525e-01, 5.0962e-04],
          [3.5673e-02, 8.1336e-01]]],


        [[[3.6564e-01, 3.0393e-01],
          [2.0941e-01, 8.0929e-01]],

         [[9.9096e-01, 5.1402e-02],
          [7.9789e-01, 5.3293e-01]]],


        [[[1.9785e-01, 9.5984e-01],
          [9.0146e-02, 4.8204e-01]],

         [[1.6335e-01, 6.6294e-01],
          [1.9594e-01, 2.2943e-02]]]])

In [85]:
mu = testX.mean(dim=(2,3), keepdim=True)
b = torch.distributions.bernoulli.Bernoulli(logits=mu).sample()

In [86]:
b.size()

torch.Size([256, 2, 1, 1])

In [87]:
b

tensor([[[[1.]],

         [[0.]]],


        [[[0.]],

         [[1.]]],


        [[[0.]],

         [[1.]]],


        [[[0.]],

         [[1.]]],


        [[[0.]],

         [[1.]]],


        [[[1.]],

         [[0.]]],


        [[[0.]],

         [[1.]]],


        [[[0.]],

         [[1.]]],


        [[[1.]],

         [[0.]]],


        [[[1.]],

         [[0.]]],


        [[[0.]],

         [[1.]]],


        [[[1.]],

         [[1.]]],


        [[[0.]],

         [[1.]]],


        [[[0.]],

         [[1.]]],


        [[[1.]],

         [[1.]]],


        [[[1.]],

         [[1.]]],


        [[[1.]],

         [[1.]]],


        [[[1.]],

         [[0.]]],


        [[[1.]],

         [[0.]]],


        [[[0.]],

         [[1.]]],


        [[[1.]],

         [[1.]]],


        [[[1.]],

         [[1.]]],


        [[[0.]],

         [[0.]]],


        [[[1.]],

         [[0.]]],


        [[[0.]],

         [[0.]]],


        [[[0.]],

         [[0.]]],


        [[[0

In [88]:
testX * b

tensor([[[[7.5499e-01, 1.7076e-01],
          [6.7209e-01, 7.8012e-02]],

         [[0.0000e+00, 0.0000e+00],
          [0.0000e+00, 0.0000e+00]]],


        [[[0.0000e+00, 0.0000e+00],
          [0.0000e+00, 0.0000e+00]],

         [[9.1916e-01, 1.6743e-01],
          [7.4843e-01, 2.5929e-01]]],


        [[[0.0000e+00, 0.0000e+00],
          [0.0000e+00, 0.0000e+00]],

         [[8.3670e-01, 8.0519e-01],
          [5.1343e-01, 6.7952e-01]]],


        ...,


        [[[5.4363e-01, 6.2376e-01],
          [6.4436e-01, 9.3477e-01]],

         [[2.2525e-01, 5.0962e-04],
          [3.5673e-02, 8.1336e-01]]],


        [[[0.0000e+00, 0.0000e+00],
          [0.0000e+00, 0.0000e+00]],

         [[0.0000e+00, 0.0000e+00],
          [0.0000e+00, 0.0000e+00]]],


        [[[0.0000e+00, 0.0000e+00],
          [0.0000e+00, 0.0000e+00]],

         [[0.0000e+00, 0.0000e+00],
          [0.0000e+00, 0.0000e+00]]]])

In [89]:
testX * (1-b) + testX * b

tensor([[[[7.5499e-01, 1.7076e-01],
          [6.7209e-01, 7.8012e-02]],

         [[4.8203e-01, 8.6753e-01],
          [3.3811e-01, 3.0815e-01]]],


        [[[5.9437e-01, 6.8330e-01],
          [5.3126e-01, 8.6268e-01]],

         [[9.1916e-01, 1.6743e-01],
          [7.4843e-01, 2.5929e-01]]],


        [[[2.4428e-01, 9.6529e-01],
          [1.7327e-01, 2.1513e-01]],

         [[8.3670e-01, 8.0519e-01],
          [5.1343e-01, 6.7952e-01]]],


        ...,


        [[[5.4363e-01, 6.2376e-01],
          [6.4436e-01, 9.3477e-01]],

         [[2.2525e-01, 5.0962e-04],
          [3.5673e-02, 8.1336e-01]]],


        [[[3.6564e-01, 3.0393e-01],
          [2.0941e-01, 8.0929e-01]],

         [[9.9096e-01, 5.1402e-02],
          [7.9789e-01, 5.3293e-01]]],


        [[[1.9785e-01, 9.5984e-01],
          [9.0146e-02, 4.8204e-01]],

         [[1.6335e-01, 6.6294e-01],
          [1.9594e-01, 2.2943e-02]]]])

In [90]:

batchsc = BatchSC()

# print(testX)
# print('after activation')
# print(batchsc(testX))

In [92]:
%%timeit -n 100 -r 10
brelu(testX)

58.6 µs ± 6.55 µs per loop (mean ± std. dev. of 10 runs, 100 loops each)


In [ ]:
%%timeit -n 100 -r 10
relu(testX)

6.43 µs ± 1.89 µs per loop (mean ± std. dev. of 10 runs, 100 loops each)


In [95]:
class ReLUPlusBRelu(pl.LightningModule):
    def __init__(self):
        super(ReLUPlusBRelu, self).__init__() #Relu + brelu
        self.relu = torch.nn.ReLU(inplace=True)
        self.brelu = BReLU()
        
    def forward(self, x):
        return self.relu(x) + self.brelu(x)

rpb = ReLUPlusBRelu()

In [96]:
rpb(testX)

tensor([[[[7.5499e-01, 3.4152e-01],
          [6.7209e-01, 7.8012e-02]],

         [[4.8203e-01, 8.6753e-01],
          [6.7622e-01, 6.1630e-01]]],


        [[[5.9437e-01, 1.3666e+00],
          [1.0625e+00, 8.6268e-01]],

         [[1.8383e+00, 3.3485e-01],
          [1.4969e+00, 5.1859e-01]]],


        [[[4.8856e-01, 1.9306e+00],
          [1.7327e-01, 2.1513e-01]],

         [[1.6734e+00, 1.6104e+00],
          [5.1343e-01, 6.7952e-01]]],


        ...,


        [[[5.4363e-01, 6.2376e-01],
          [1.2887e+00, 1.8695e+00]],

         [[4.5051e-01, 5.0962e-04],
          [3.5673e-02, 1.6267e+00]]],


        [[[7.3129e-01, 3.0393e-01],
          [4.1883e-01, 8.0929e-01]],

         [[1.9819e+00, 5.1402e-02],
          [1.5958e+00, 1.0659e+00]]],


        [[[1.9785e-01, 1.9197e+00],
          [1.8029e-01, 4.8204e-01]],

         [[3.2670e-01, 1.3259e+00],
          [3.9188e-01, 2.2943e-02]]]])

In [ ]:
ib = ImproveBatchWiseSARBSC()

In [ ]:
%%timeit -n 100 -r 10
ib(testX)

172 µs ± 6.72 µs per loop (mean ± std. dev. of 10 runs, 100 loops each)


In [ ]:
relu = torch.nn.ReLU()
with Timer():
    relu(testX)

0.00011444091796875


In [ ]:
batchwise = BatchWiseSARBSC()
with Timer():
    batchwise(testX)

0.010975360870361328


In [ ]:
tx = torch.randint(-5, 5, (4, 3, 2, 2)).to(torch.float32)
# ty = []
# for i in tx:
#     print(i-1)
#     ty.append(i-1)
# ty = torch.stack(ty)
print(tx)
# print(ty)
mu = tx.mean(dim=(1, 2, 3), keepdim=False)
print(mu.shape)
# print(torch.distributions.bernoulli.Bernoulli(logits=tx.mean(dim=(1,2, 3), keepdim=True)).sample().expand_as(tx))
p = -torch.sin(mu) * torch.cos(mu + (torch.pi/2))
p = torch.where(p < 0, 0, p)
p = torch.where(p > 1, 1, p)
b = torch.distributions.bernoulli.Bernoulli(probs=p).sample()
print(b)
for switch, channel in zip(b, tx):
    print(f'switch = {switch} with channel = {channel}')

tensor([[[[ 0.,  4.],
          [ 3.,  4.]],

         [[ 4., -4.],
          [-1., -3.]],

         [[ 1.,  2.],
          [-4.,  2.]]],


        [[[-1.,  2.],
          [-5.,  0.]],

         [[-5.,  4.],
          [ 4.,  1.]],

         [[-5.,  0.],
          [-4., -4.]]],


        [[[-2., -2.],
          [ 1.,  0.]],

         [[-5., -1.],
          [ 3.,  1.]],

         [[-5., -5.],
          [-3.,  1.]]],


        [[[ 3.,  4.],
          [ 1., -5.]],

         [[-5., -5.],
          [-3., -3.]],

         [[ 4., -4.],
          [-4.,  4.]]]])
torch.Size([4])
tensor([0., 1., 1., 0.])
switch = 0.0 with channel = tensor([[[ 0.,  4.],
         [ 3.,  4.]],

        [[ 4., -4.],
         [-1., -3.]],

        [[ 1.,  2.],
         [-4.,  2.]]])
switch = 1.0 with channel = tensor([[[-1.,  2.],
         [-5.,  0.]],

        [[-5.,  4.],
         [ 4.,  1.]],

        [[-5.,  0.],
         [-4., -4.]]])
switch = 1.0 with channel = tensor([[[-2., -2.],
         [ 1.,  0.]],

        

## 확률 식으로 sin, cos 사용

In [ ]:
class BwithSinCos(pl.LightningModule):
    def __init__(self):
        super(BwithSinCos, self).__init__()
        
    def forward(self, x):
        p = -torch.sin(x) * torch.cos(x + (torch.pi/2))
        print('out of 1')
        print((p > 1).nonzero(as_tuple=True))
        print(p[(p > 1).nonzero(as_tuple=True)])
        print('out of 0')
        print((p < 0 ).nonzero(as_tuple=True))
        print(p[(p < 0 ).nonzero(as_tuple=True)])
        print()
        epsilon = torch.distributions.bernoulli.Bernoulli(probs=p).sample()
        return x * epsilon.to(self.device)

In [ ]:
sc = BwithSinCos()
sc(x)

out of 1
(tensor([], dtype=torch.int64), tensor([], dtype=torch.int64))
tensor([])
out of 0
(tensor([], dtype=torch.int64), tensor([], dtype=torch.int64))
tensor([])



tensor([[ -0.,  -0., -11.],
        [  0.,  -0.,  -0.],
        [ -5.,  -7.,   1.]])

In [ ]:
i = torch.tensor([[1.56, torch.pi/2],
                  [0, 1]])

sc(i)

out of 1
(tensor([], dtype=torch.int64), tensor([], dtype=torch.int64))
tensor([])
out of 0
(tensor([], dtype=torch.int64), tensor([], dtype=torch.int64))
tensor([])



tensor([[1.5600, 1.5708],
        [0.0000, 0.0000]])

## 얽힘 (구현 방법 생각 중)

모든 칸을 돌면서 자기 주위의 1 개수를 센 다음 % 2 한 값을 자기 위치에 갖는다.

In [ ]:
b = torch.distributions.bernoulli.Bernoulli(logits=x).sample()
o = torch.zeros_like(b)
print(b)
print(o)

tensor([[0., 0., 0.],
        [1., 0., 0.],
        [0., 0., 1.]])
tensor([[0., 0., 0.],
        [0., 0., 0.],
        [0., 0., 0.]])


In [ ]:
a = torch.randint(0, 2, (3, 4, 5))
a

tensor([[[0, 0, 1, 0, 1],
         [1, 1, 1, 1, 1],
         [0, 0, 0, 0, 1],
         [1, 0, 0, 0, 0]],

        [[0, 1, 1, 1, 0],
         [0, 0, 1, 1, 1],
         [0, 1, 0, 1, 1],
         [1, 1, 1, 1, 1]],

        [[1, 0, 1, 0, 1],
         [1, 1, 0, 0, 0],
         [1, 1, 0, 1, 0],
         [0, 0, 1, 0, 0]]])

In [ ]:
(a==1).nonzero(as_tuple=True)
a[(a==1).nonzero(as_tuple=True)] += 1 

In [ ]:
(a==1).nonzero(as_tuple=True)

(tensor([], dtype=torch.int64),
 tensor([], dtype=torch.int64),
 tensor([], dtype=torch.int64))

In [ ]:
a

tensor([[[0, 0, 2, 0, 2],
         [2, 2, 2, 2, 2],
         [0, 0, 0, 0, 2],
         [2, 0, 0, 0, 0]],

        [[0, 2, 2, 2, 0],
         [0, 0, 2, 2, 2],
         [0, 2, 0, 2, 2],
         [2, 2, 2, 2, 2]],

        [[2, 0, 2, 0, 2],
         [2, 2, 0, 0, 0],
         [2, 2, 0, 2, 0],
         [0, 0, 2, 0, 0]]])

In [ ]:
ones = (b==1).nonzero(as_tuple=False)

for i in ones:
    print(i)
    print(b[tuple(i)])

tensor([1, 0])
tensor(1.)
tensor([2, 2])
tensor(1.)


-------------------------------------

In [ ]:
f = torch.distributions.bernoulli.Bernoulli(logits=alpha * x)

In [ ]:
f.probs

tensor([[0.2689, 0.3543, 0.2497],
        [0.7858, 0.2315, 0.3543],
        [0.3775, 0.3318, 0.5250]])

In [ ]:
e = torch.exp(alpha * x) / (torch.exp(alpha * x) + 1)
print(e)

tensor([[0.2689, 0.3543, 0.2497],
        [0.7858, 0.2315, 0.3543],
        [0.3775, 0.3318, 0.5250]])


In [ ]:
p = -torch.sin(x) * torch.cos(x+ (torch.pi/2))
print(p)
torch.distributions.bernoulli.Bernoulli(probs=p).sample()

tensor([[0.2960, 0.0781, 1.0000],
        [0.1765, 0.2879, 0.0781],
        [0.9195, 0.4316, 0.7081]])


tensor([[0., 0., 1.],
        [0., 0., 0.],
        [1., 0., 0.]])